<a href="https://colab.research.google.com/github/FaishalFaruq02/DataAnalyst-Nasabah-Churn/blob/main/churn_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import numpy as np
import pandas as pd

In [17]:
df = pd.read_csv("/content/bank_churn_dirty.csv")

df_original = df.copy()
print("Jumlah baris:", len(df))

display(df.head())

Jumlah baris: 10150


,customer_id,nome,email,numero_conta,data_nascimento,cidade,country,gender,age,credit_score,tenure,balance,limite_credito,products_number,credit_card,active_member,estimated_salary,churn
0,15631460.0,Camila Martins Lima,camila.martins@hotmail.com,83406-0,04/10/1982,Barcelona,Spain,Female,42,671,3,0.00,0.00,2,1,1,128449.33,0
1,15700300.0,Patrícia Rocha Alves,patricia.rocha@uol.com.br,83680-0,18/12/1980,Berlin,Germany,Female,44,674,4,131593.85,202324.81,1,0,1,171345.02,1
2,NaN,ANDRÉ COSTA ARAÚJO,andre.costa@gmail.com,73884-8,1985-08-26,Berlin,Germany,Male,39,789,7,124828.46,356669.15,2,1,1,124411.08,0
3,15760456.0,Silvana Costa Ferreira,silvana.costa@outlook.com,43299-9,1986-04-04,Paris,France,Female,38,731,10,123711.73,262028.34,2,1,0,171340.68,1
4,15638730.0,Natália Santos Martins,natalia.santos@yahoo.com.br,56717-2,2003-04-11,Paris,France,Female,-1,711,0,82844.33,207795.24,2,0,1,1408.68,0


In [18]:
# hapus duplikat
df = df.drop_duplicates().copy()

print("\n--- Pemeriksaan Duplikat Baris ---")
print("Baris sebelum:", len(df_original))
print("Baris sesudah:", len(df))
print("Duplikat baris dihapus:", len(df_original) - len(df))


--- Pemeriksaan Duplikat Baris ---
Baris sebelum: 10150
Baris sesudah: 10001
Duplikat baris dihapus: 149


In [19]:
# rename kolom
df = df.rename(columns={
    "cidade": "city",
    "limite_credito": "limit_credit"
})

In [20]:
# catat masalah sebelum memperbaiki nilainya
quality_flag = pd.Series("", index=df.index, dtype="object")

def add_flag(mask, label):
    """Tambahkan label masalah pada baris yang sesuai."""
    mask = mask.fillna(False)
    quality_flag.loc[mask] = quality_flag.loc[mask].apply(
        lambda current: (
            f"{current}; {label}" if current else label
        )
    )

# Pastikan kolom numerik benar-benar numerik
numeric_cols = [
    "customer_id", "age", "credit_score", "balance",
    "estimated_salary", "churn"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Catat masalah ID sebelum memperbaiki ID
add_flag(df["customer_id"].isna(), "customer_id kosong")
add_flag(
    df["customer_id"].notna()
    & df["customer_id"].duplicated(keep="first"),
    "customer_id duplikat"
)

# Catat masalah data finansial
negative_balance = df["balance"] < 0

placeholder_balance = df["balance"].isin([
    999999.99,
    99_999_999
])

invalid_score = (
    (df["credit_score"] < 300)
    | (df["credit_score"] > 850)
)

zero_salary = df["estimated_salary"] == 0
missing_salary = df["estimated_salary"].isna()

add_flag(negative_balance, "saldo negatif")
add_flag(placeholder_balance, "saldo placeholder")
add_flag(invalid_score, "credit score di luar rentang 300-850")
add_flag(zero_salary, "estimated salary nol")
add_flag(missing_salary, "estimated salary kosong")

In [21]:
# olah dan perbaiki usia
old_age = df["age"].copy()

dob = pd.to_datetime(
    df["data_nascimento"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Gunakan tanggal acuan yang konsisten
today = pd.Timestamp("2027-01-01")

calculated_age = (
    today.year
    - dob.dt.year
    - (
        (today.month < dob.dt.month)
        | (
            (today.month == dob.dt.month)
            & (today.day < dob.dt.day)
        )
    ).astype(int)
)

# Gunakan usia hasil perhitungan jika valid.
# Jika tanggal lahir tidak valid, gunakan usia lama jika valid.
valid_calculated_age = calculated_age.between(18, 200)
valid_old_age = old_age.between(18, 200)

df["age"] = calculated_age.where(
    valid_calculated_age,
    old_age.where(valid_old_age, np.nan)
)

add_flag(
    df["age"].isna() | ~df["age"].between(18, 200),
    "usia tidak valid"
)

In [22]:
# normalisasi
for col in ["city", "country", "gender"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.title()
        )

In [23]:
# perbaiki customer id
used_ids = set(
    df.loc[
        df["customer_id"].notna(),
        "customer_id"
    ].astype("int64")
)

next_id = 600_000_000
new_ids = []
seen_ids = set()

for value in df["customer_id"]:
    if pd.isna(value) or int(value) in seen_ids:
        while next_id in used_ids or next_id in seen_ids:
            next_id += 1

        new_ids.append(next_id)
        seen_ids.add(next_id)
        used_ids.add(next_id)
        next_id += 1
    else:
        value = int(value)
        new_ids.append(value)
        seen_ids.add(value)

df["customer_id"] = pd.Series(
    new_ids, index=df.index, dtype="int64"
)

In [24]:
# bersihkan nilai tidak valid
# baris dipertahankan, nilai bermasalah menjadi NaN
df.loc[negative_balance, "balance"] = np.nan
df.loc[placeholder_balance, "balance"] = np.nan
df.loc[invalid_score, "credit_score"] = np.nan
df.loc[zero_salary, "estimated_salary"] = np.nan

In [25]:
# feature engineering
def age_group(age):
    if pd.isna(age):
        return "Tidak Valid"
    if age < 30:
        return "<30"
    if age < 40:
        return "30-39"
    if age < 50:
        return "40-49"
    if age < 60:
        return "50-59"
    return "60+"


def score_band(score):
    if pd.isna(score):
        return "Tidak Valid"
    if score < 580:
        return "Poor (<580)"
    if score < 670:
        return "Fair (580-669)"
    if score < 740:
        return "Good (670-739)"
    if score < 800:
        return "Very Good (740-799)"
    return "Excellent (800+)"


def balance_segment(balance):
    if pd.isna(balance):
        return "Tidak Valid"
    if balance == 0:
        return "Nol"
    if balance < 800_000:
        return "<8 jt"
    if balance < 1_200_000:
        return "8-12 jt"
    if balance < 1_600_000:
        return "12-16 jt"
    return ">16 jt"


df["age_group"] = df["age"].apply(age_group)
df["score_band"] = df["credit_score"].apply(score_band)
df["balance_segment"] = df["balance"].apply(balance_segment)

df["churn_label"] = (
    df["churn"]
    .map({0: "Bertahan", 1: "Churn"})
    .fillna("Tidak Valid")
)

In [26]:
# buat data quality flag
df["data_quality_flag"] = quality_flag.replace("", "OK")

In [27]:
# hapus kolom yg tidak dipakai
drop_cols = [
    "nome",
    "email",
    "numero_conta",
    "data_nascimento"
]

df = df.drop(
    columns=[col for col in drop_cols if col in df.columns]
)

In [28]:
# atur urutan kolom
columns = [
    "customer_id", "city", "country", "gender",
    "age", "age_group",
    "credit_score", "score_band",
    "tenure", "balance", "balance_segment",
    "limit_credit", "products_number",
    "credit_card", "active_member",
    "estimated_salary", "churn", "churn_label",
    "data_quality_flag"
]

df = df[
    [col for col in columns if col in df.columns]
    + [col for col in df.columns if col not in columns]
]

In [29]:
display(df.head())

,customer_id,city,country,gender,age,age_group,credit_score,score_band,tenure,balance,balance_segment,limit_credit,products_number,credit_card,active_member,estimated_salary,churn,churn_label,data_quality_flag
0,15631460,Barcelona,Spain,Female,44,40-49,671.0,Good (670-739),3,0.00,Nol,0.00,2,1,1,128449.33,0,Bertahan,OK
1,15700300,Berlin,Germany,Female,46,40-49,674.0,Good (670-739),4,131593.85,<8 jt,202324.81,1,0,1,171345.02,1,Churn,OK
2,600000000,Berlin,Germany,Male,41,40-49,789.0,Very Good (740-799),7,124828.46,<8 jt,356669.15,2,1,1,124411.08,0,Bertahan,customer_id kosong
3,15760456,Paris,France,Female,40,40-49,731.0,Good (670-739),10,123711.73,<8 jt,262028.34,2,1,0,171340.68,1,Churn,OK
4,15638730,Paris,France,Female,23,<30,711.0,Good (670-739),0,82844.33,<8 jt,207795.24,2,0,1,1408.68,0,Bertahan,OK


In [30]:
# validasi hasil
print("\n--- VALIDASI HASIL CLEANING ---")
print("Jumlah baris akhir:", len(df))
print("Jumlah kolom akhir:", len(df.columns))
print(
    "Duplikat customer_id:",
    df["customer_id"].duplicated().sum()
)
print(
    "\nJumlah baris berdasarkan data_quality_flag:"
)
print(df["data_quality_flag"].value_counts())

print("\nMissing values per kolom:")
print(df.isna().sum())

display(df.head())


--- VALIDASI HASIL CLEANING ---
Jumlah baris akhir: 10001
Jumlah kolom akhir: 19
Duplikat customer_id: 0

Jumlah baris berdasarkan data_quality_flag:
data_quality_flag
OK                                                                              8473
estimated salary kosong                                                          555
saldo negatif                                                                    285
credit score di luar rentang 300-850                                             256
customer_id kosong                                                               178
saldo placeholder                                                                173
credit score di luar rentang 300-850; estimated salary kosong                     16
saldo placeholder; credit score di luar rentang 300-850                           11
saldo placeholder; estimated salary kosong                                        10
saldo negatif; credit score di luar rentang 300-850               

,customer_id,city,country,gender,age,age_group,credit_score,score_band,tenure,balance,balance_segment,limit_credit,products_number,credit_card,active_member,estimated_salary,churn,churn_label,data_quality_flag
0,15631460,Barcelona,Spain,Female,44,40-49,671.0,Good (670-739),3,0.00,Nol,0.00,2,1,1,128449.33,0,Bertahan,OK
1,15700300,Berlin,Germany,Female,46,40-49,674.0,Good (670-739),4,131593.85,<8 jt,202324.81,1,0,1,171345.02,1,Churn,OK
2,600000000,Berlin,Germany,Male,41,40-49,789.0,Very Good (740-799),7,124828.46,<8 jt,356669.15,2,1,1,124411.08,0,Bertahan,customer_id kosong
3,15760456,Paris,France,Female,40,40-49,731.0,Good (670-739),10,123711.73,<8 jt,262028.34,2,1,0,171340.68,1,Churn,OK
4,15638730,Paris,France,Female,23,<30,711.0,Good (670-739),0,82844.33,<8 jt,207795.24,2,0,1,1408.68,0,Bertahan,OK


In [31]:
# export ke excel
output_path = "/content/Bank_Churn_Cleaned_Python.xlsx"

df.to_excel(output_path, index=False)

print("\nFile berhasil disimpan:", output_path)


File berhasil disimpan: /content/Bank_Churn_Cleaned_Python.xlsx
